# 01 — Exploratory Data Analysis
**Purpose:** Understand the dataset structure and quality before any root cause analysis is performed. This notebook produces every table referenced in the README's Business Context and Dataset Overview sections.

No hypothesis testing or root cause analysis is performed here — see `02_Root_Cause_Analysis.ipynb` for that.


In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 25)

df = pd.read_excel('/content/superstore_data1.xlsx', sheet_name='Sample - Superstore')
print("Shape:", df.shape)


Shape: (9994, 21)


## 1. Data Loading & Schema

**Business question:** What does the dataset contain, and at what granularity?


In [5]:
schema = pd.DataFrame({
    'column': df.columns,
    'dtype': df.dtypes.astype(str).values,
    'n_unique': [df[c].nunique() for c in df.columns],
    'example_value': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
schema


,column,dtype,n_unique,example_value
0,Row ID,int64,9994,1
1,Order ID,object,5009,CA-2016-152156
2,Order Date,datetime64[ns],1237,2016-11-08 00:00:00
3,Ship Date,datetime64[ns],1334,2016-11-11 00:00:00
4,Ship Mode,object,4,Second Class
5,Customer ID,object,793,CG-12520
6,Customer Name,object,793,Claire Gute
7,Segment,object,3,Consumer
8,Country,object,1,United States
9,City,object,531,Henderson


**README claim check:** "Rows: 9,994 / Columns: 21 / Granularity: one row per order line item"


In [6]:
print("Total rows:", len(df))
print("Total columns:", df.shape[1])
print("Unique Order IDs:", df['Order ID'].nunique(), "-> multiple rows per order confirms line-item granularity")


Total rows: 9994
Total columns: 21
Unique Order IDs: 5009 -> multiple rows per order confirms line-item granularity


## 2. Data Quality Assessment

**Business question:** Is the dataset reliable enough to support the analysis?


In [7]:
print("=== Missing values per column ===")
print(df.isnull().sum())
print()
print("Total missing values across dataset:", df.isnull().sum().sum())


=== Missing values per column ===
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Total missing values across dataset: 0


In [8]:
print("=== Duplicate rows ===")
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate Row ID:", df['Row ID'].duplicated().sum())
dup_mask = df.duplicated(subset=['Order ID','Product ID'], keep=False)
print("Rows sharing identical (Order ID, Product ID):", dup_mask.sum())


=== Duplicate rows ===
Fully duplicated rows: 0
Duplicate Row ID: 0
Rows sharing identical (Order ID, Product ID): 16


In [9]:
print("=== Invalid value checks ===")
print("Negative Sales:", (df['Sales']<0).sum())
print("Negative Quantity:", (df['Quantity']<0).sum())
print("Zero Quantity:", (df['Quantity']==0).sum())
print("Discount range: [%.2f, %.2f]" % (df['Discount'].min(), df['Discount'].max()))
print("Discount outside [0,1]:", ((df['Discount']<0)|(df['Discount']>1)).sum())


=== Invalid value checks ===
Negative Sales: 0
Negative Quantity: 0
Zero Quantity: 0
Discount range: [0.00, 0.80]
Discount outside [0,1]: 0


In [10]:
print("=== Constant columns ===")
for c in df.columns:
    if df[c].nunique() == 1:
        print(f"{c}: constant value = {df[c].unique()}")


=== Constant columns ===
Country: constant value = ['United States']


## 3. Time Coverage


In [11]:
print("Order Date range:", df['Order Date'].min(), "to", df['Order Date'].max())
print("Ship Date range:", df['Ship Date'].min(), "to", df['Ship Date'].max())
print("Rows with Ship Date < Order Date:", (df['Ship Date'] < df['Order Date']).sum())


Order Date range: 2014-01-03 00:00:00 to 2017-12-30 00:00:00
Ship Date range: 2014-01-07 00:00:00 to 2018-01-05 00:00:00
Rows with Ship Date < Order Date: 0


## 4. Summary Statistics — Sales, Profit, Margin


In [12]:
summary_stats = df[['Sales','Quantity','Discount','Profit']].describe()
summary_stats


,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000
mean,229.858001,3.789574,0.156203,28.656896
std,623.245101,2.225110,0.206452,234.260108
min,0.444000,1.000000,0.000000,-6599.978000
25%,17.280000,2.000000,0.000000,1.728750
50%,54.490000,3.000000,0.200000,8.666500
75%,209.940000,5.000000,0.200000,29.364000
max,22638.480000,14.000000,0.800000,8399.976000


In [13]:
df['MarginRow'] = df['Profit'] / df['Sales']
print("Overall dataset Profit Margin (Total Profit / Total Sales): %.4f%%" % (df['Profit'].sum()/df['Sales'].sum()*100))
print("Negative-profit line items: %d (%.2f%% of rows)" % ((df['Profit']<0).sum(), (df['Profit']<0).mean()*100))


Overall dataset Profit Margin (Total Profit / Total Sales): 12.4672%
Negative-profit line items: 1871 (18.72% of rows)


## 5. Category Overview


In [14]:
category_overview = df.groupby('Category').agg(
    Rows=('Row ID','count'),
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
)
category_overview['Margin%'] = category_overview['Profit']/category_overview['Revenue']*100
category_overview['RevenueShare%'] = category_overview['Revenue']/category_overview['Revenue'].sum()*100
category_overview


,Rows,Revenue,Profit,Margin%,RevenueShare%
Category,,,,,
Furniture,2121,741999.7953,18451.2728,2.486695,32.300171
Office Supplies,6026,719047.0320,122490.8008,17.035158,31.301008
Technology,1847,836154.0330,145454.9481,17.395712,36.398821


In [15]:
subcategory_overview = df.groupby(['Category','Sub-Category']).agg(
    Rows=('Row ID','count'),
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
)
subcategory_overview['Margin%'] = subcategory_overview['Profit']/subcategory_overview['Revenue']*100
subcategory_overview


Rows      Revenue      Profit    Margin%
Category        Sub-Category                                          
Furniture       Bookcases      228  114879.9963  -3472.5560  -3.022768
                Chairs         617  328449.1030  26590.1663   8.095673
                Furnishings    957   91705.1640  13059.1436  14.240358
                Tables         319  206965.5320 -17725.4811  -8.564460
Office Supplies Appliances     466  107532.1610  18138.0054  16.867517
                Art            796   27118.7920   6527.7870  24.071083
                Binders       1523  203412.7330  30221.7633  14.857361
                Envelopes      254   16476.4020   6964.1767  42.267582
                Fasteners      217    3024.2800    949.5182  31.396504
                Labels         364   12486.3120   5546.2540  44.418672
                Paper         1370   78479.2060  34053.5693  43.391837
                Storage        846  223843.6080  21278.8264   9.506113
                Supplies       190   46673.5380  -1189.0995  -2.547695
Technology      Accessories    775  167380.3180  41936.6357  25.054700
                Copiers         68  149528.0300  55617.8249  37.195585
                Machines       115  189238.6310   3384.7569   1.788618
                Phones         889  330007.0540  44515.7306  13.489327

## 6. Region Overview


In [16]:
region_overview = df.groupby('Region').agg(
    Rows=('Row ID','count'),
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
)
region_overview['Margin%'] = region_overview['Profit']/region_overview['Revenue']*100
region_overview['RevenueShare%'] = region_overview['Revenue']/region_overview['Revenue'].sum()*100
region_overview


,Rows,Revenue,Profit,Margin%,RevenueShare%
Region,,,,,
Central,2323,501239.8908,39706.3625,7.921629,21.819594
East,2848,678781.2400,91522.7800,13.483399,29.548188
South,1620,391721.9050,46749.4303,11.934342,17.052140
West,3203,725457.8245,108418.4489,14.944831,31.580078


## 7. Segment Overview


In [17]:
segment_overview = df.groupby('Segment').agg(
    Rows=('Row ID','count'),
    Revenue=('Sales','sum'),
    Profit=('Profit','sum')
)
segment_overview['Margin%'] = segment_overview['Profit']/segment_overview['Revenue']*100
segment_overview['RevenueShare%'] = segment_overview['Revenue']/segment_overview['Revenue'].sum()*100
segment_overview


,Rows,Revenue,Profit,Margin%,RevenueShare%
Segment,,,,,
Consumer,5191,1.161401e+06,134119.2092,11.548050,50.557240
Corporate,3020,7.061464e+05,91979.1340,13.025506,30.739426
Home Office,1783,4.296531e+05,60298.6785,14.034269,18.703334


## 8. Entity Overview (supports "Entities" line in README Dataset Overview)


In [18]:
print("Unique Customers:", df['Customer ID'].nunique())
print("Unique Products:", df['Product ID'].nunique())
print("Unique Orders:", df['Order ID'].nunique())
print("Countries:", df['Country'].unique())
print("Regions:", sorted(df['Region'].unique()))
print("Segments:", sorted(df['Segment'].unique()))
print("Categories:", sorted(df['Category'].unique()))


Unique Customers: 793
Unique Products: 1862
Unique Orders: 5009
Countries: ['United States']
Regions: ['Central', 'East', 'South', 'West']
Segments: ['Consumer', 'Corporate', 'Home Office']
Categories: ['Furniture', 'Office Supplies', 'Technology']


## Summary

This notebook confirms:
- Dataset shape: 9,994 rows × 21 columns, one row per order line item.
- No missing values in any column.
- No invalid Sales/Quantity/Discount values.
- `Country` is a constant column (United States only).
- Data spans January 2014 – December 2017.
- 3 Categories, 4 Regions, 3 Segments — matching the README Dataset Overview section.

Root cause analysis begins in `02_Root_Cause_Analysis.ipynb`.
